# MatchFormer Epipolar Fine-tuning — Google Colab

This notebook fine-tunes MatchFormer with epipolar supervision baked in.  
Checkpoints are saved to **Google Drive** so training can be resumed if Colab disconnects.

**Before running:**
1. Set Runtime → Change runtime type → **T4 GPU**
2. Upload your ScanNet data to Google Drive (see Cell 3)
3. Run cells **top to bottom** — on resume, skip to Cell 6

## Cell 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# All checkpoints will be saved here — survives Colab restarts
DRIVE_CKPT_DIR = '/content/drive/MyDrive/matchformer_checkpoints'

import os
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
print(f'Checkpoint dir: {DRIVE_CKPT_DIR}')

## Cell 2: Clone Repo & Install Dependencies

In [ ]:
import os

REPO_URL = 'https://github.com/sid-raj-uc/match-former.git'  # your repo
REPO_DIR = '/content/match-former'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned — pulling latest')
    !cd {REPO_DIR} && git pull

%cd /content/match-former/MatchFormer

In [ ]:
# Install dependencies
!pip install -q pytorch-lightning einops kornia timm loguru yacs
!pip install -q opencv-python-headless
print('Dependencies installed.')

## Cell 3: Upload ScanNet Data

Upload your exported ScanNet scene to Google Drive. Expected structure:
```
MyDrive/scannet_data/scene0000_00/exported/
    color/    *.jpg
    depth/    *.png
    pose/     *.txt
    intrinsic/intrinsic_depth.txt
```
Or zip and upload, then unzip below.

In [ ]:
# Option A: Data already on Drive — just set the path
DATA_DIR = '/content/drive/MyDrive/scannet_data/scene0000_00/exported'

# Option B: Unzip from Drive
# ZIP_PATH = '/content/drive/MyDrive/scannet_scene0000_00.zip'
# !unzip -q {ZIP_PATH} -d /content/scannet_data/
# DATA_DIR = '/content/scannet_data/scene0000_00/exported'

# Verify
import os, glob
n_color = len(glob.glob(os.path.join(DATA_DIR, 'color', '*.jpg')))
n_depth = len(glob.glob(os.path.join(DATA_DIR, 'depth', '*.png')))
print(f'Found {n_color} color frames, {n_depth} depth frames')

## Cell 4: Download Pretrained Weights

In [ ]:
# Check if pretrained weight already exists (from a previous run or Drive)
WEIGHTS_PATH = 'model/weights/indoor-lite-LA.ckpt'
DRIVE_WEIGHTS = '/content/drive/MyDrive/matchformer_checkpoints/indoor-lite-LA.ckpt'

os.makedirs('model/weights', exist_ok=True)

if os.path.exists(DRIVE_WEIGHTS) and not os.path.exists(WEIGHTS_PATH):
    import shutil
    shutil.copy(DRIVE_WEIGHTS, WEIGHTS_PATH)
    print('Copied weights from Drive')
elif os.path.exists(WEIGHTS_PATH):
    print('Weights already present')
else:
    print('⚠️  Please upload indoor-lite-LA.ckpt to Drive at:', DRIVE_WEIGHTS)
    print('   or place it at:', WEIGHTS_PATH)

## Cell 5: Verify GPU & Run Sanity Check
Run this once to confirm the pipeline works before doing the full training.

In [ ]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (no GPU!)"}')
print(f'CUDA available: {torch.cuda.is_available()}')

In [ ]:
# Sanity check: 5 pairs, 50 steps — should finish in ~2 min on T4
!python train_finetune.py \
    --overfit \
    --data_dir {DATA_DIR} \
    --checkpoint_dir {DRIVE_CKPT_DIR}/overfit \
    --steps 50

print('Sanity check complete — loss should be decreasing!')

## Cell 6: Full Fine-tuning

**Auto-resume is enabled** — if Colab disconnects, just re-run from Cell 1 and then run this cell again. It will automatically pick up from the last saved checkpoint in your Drive.

**Key hyperparameters:**
- `--steps 10000` — total training steps (~2-3 hours on T4)
- `--tau 10.0` — epipolar constraint strength
- `--save_every 200` — save checkpoint to Drive every 200 steps
- `--batch 4` — batch size (reduce to 2 if OOM)

In [ ]:
!python train_finetune.py \
    --data_dir {DATA_DIR} \
    --checkpoint_dir {DRIVE_CKPT_DIR} \
    --steps 10000 \
    --tau 10.0 \
    --batch 4 \
    --workers 4 \
    --lr 1e-4 \
    --save_every 200

# To resume manually from a specific checkpoint:
# !python train_finetune.py \
#     --data_dir {DATA_DIR} \
#     --checkpoint_dir {DRIVE_CKPT_DIR} \
#     --resume {DRIVE_CKPT_DIR}/last.ckpt \
#     --steps 10000 \
#     --tau 10.0 --batch 4 --save_every 200

## Cell 7: Run Benchmark After Training

In [ ]:
# After training finishes, copy the best checkpoint locally and run benchmark
import shutil

TRAINED_CKPT = f'{DRIVE_CKPT_DIR}/last.ckpt'
shutil.copy(TRAINED_CKPT, 'model/weights/epipolar-finetuned.ckpt')
print('Checkpoint copied.')

# Run benchmark with finetuned model
!python run_benchmark.py \
    --num_pairs 100 \
    --ckpt model/weights/epipolar-finetuned.ckpt \
    2>&1 | tee benchmark_finetuned_log.txt

# Copy results back to Drive
shutil.copy('benchmark_finetuned_log.txt', f'{DRIVE_CKPT_DIR}/benchmark_finetuned_log.txt')
print('Benchmark results saved to Drive!')